# Music Artist Data — Exploratory Analysis
Loading and exploring the Viberate artist chart data across 5 genres.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from music.config import GENRES_DIR, GENRE_NAMES

## Load genre artist charts

In [ ]:
frames = []
for genre in GENRE_NAMES:
    path = GENRES_DIR / genre / f"{genre}-Artist_Chart-Current_Total.csv"
    df = pd.read_csv(path)
    df["Genre"] = genre
    frames.append(df)

artists = pd.concat(frames, ignore_index=True)
print(artists.shape)

## First look

In [ ]:
artists.head()

In [ ]:
artists.dtypes

In [ ]:
artists.describe()

## Missing values

In [ ]:
artists.isnull().sum().sort_values(ascending=False).head(15)

## Artists per genre

In [ ]:
artists.groupby("Genre").size()

## Top 10 artists by Spotify streams per genre

In [ ]:
import plotly.express as px

top10 = (
    artists.dropna(subset=["Spotify Streams Total"])
    .groupby("Genre", group_keys=False)
    .apply(lambda g: g.nlargest(10, "Spotify Streams Total"))
    .reset_index(drop=True)
)

fig = px.bar(
    top10,
    x="Artist Name",
    y="Spotify Streams Total",
    color="Genre",
    facet_col="Genre",
    facet_col_wrap=3,
    title="Top 10 Artists by Total Spotify Streams per Genre",
    labels={"Spotify Streams Total": "Total Streams", "Artist Name": ""},
)
fig.update_xaxes(matches=None, showticklabels=True, tickangle=45)
fig.update_yaxes(matches=None)
fig.update_layout(showlegend=False, height=700)
fig.show()

## Cross-platform correlation heatmap

In [ ]:
cols = [
    "Spotify Streams Total",
    "Spotify Followers",
    "Spotify Monthly Listeners",
    "YouTube Views",
    "TikTok Followers",
    "Instagram Followers",
    "Facebook Followers",
]

corr = artists[cols].dropna().corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Cross-Platform Metric Correlations",
)
fig.update_layout(height=550)
fig.show()

## Top artists by country

In [ ]:
from music.config import COUNTRIES_DIR

country_frames = []
for country_dir in sorted(COUNTRIES_DIR.iterdir()):
    csv_files = list(country_dir.glob("*_top_artists.csv"))
    if csv_files:
        df = pd.read_csv(csv_files[0])
        df["Country"] = country_dir.name
        country_frames.append(df)

country_artists = pd.concat(country_frames, ignore_index=True)

counts = (
    country_artists.groupby("Country")
    .size()
    .reset_index(name="Artist Count")
    .sort_values("Artist Count", ascending=False)
)

fig = px.bar(
    counts,
    x="Country",
    y="Artist Count",
    title="Number of Top Artists per Country",
    color="Artist Count",
    color_continuous_scale="Blues",
)
fig.update_layout(coloraxis_showscale=False, xaxis_tickangle=45, height=500)
fig.show()

## Coachella subgenre distribution

In [ ]:
from music.config import FESTIVALS_DIR

coachella_path = FESTIVALS_DIR / "Coachella" / "coachella_subgenre_distribution_apr.csv"
coachella = pd.read_csv(coachella_path)

top_subgenres = (
    coachella.groupby("Subgenre", as_index=False)["Points"]
    .sum()
    .nlargest(15, "Points")
)

fig = px.bar(
    top_subgenres,
    x="Points",
    y="Subgenre",
    orientation="h",
    title="Coachella — Top 15 Subgenres by Streaming Points",
    color="Points",
    color_continuous_scale="Purples",
)
fig.update_layout(coloraxis_showscale=False, yaxis={"categoryorder": "total ascending"}, height=500)
fig.show()

## Key Findings

**Genre & Spotify Streams**
- Pop and Hip Hop artists dominate total Spotify stream counts, with the top artists in these genres far outpacing other genres.
- Metal and Rock show more evenly distributed streams across artists, with fewer extreme outliers.

**Cross-Platform Correlations**
- Spotify metrics (followers, monthly listeners, total streams) are strongly correlated with each other — artists who are big on one Spotify metric tend to be big on all.
- Cross-platform correlation (e.g. Spotify vs TikTok) is weaker, suggesting that streaming dominance does not always translate to social platform popularity.
- YouTube Views shows moderate correlation with Spotify, while TikTok and Instagram follow counts behave more independently.

**Country Distribution**
- The US and UK contribute the most top-charting artists by a wide margin.
- Latin American countries (Brazil, Mexico, Colombia, Argentina) are well represented, reflecting the global rise of Latin music.
- Emerging markets like Nigeria and Indonesia are present, highlighting the growing global footprint of local music scenes.

**Coachella Subgenre Distribution**
- Mainstream Pop and Indie Pop lead Coachella's lineup by streaming points, confirming the festival's mainstream lean.
- Hip Hop subgenres (Alternative Hip Hop, Contemporary Hip Hop) are strongly represented alongside Pop.
- Niche subgenres have low individual point totals but collectively make up a significant share of the lineup diversity.